# Bước 4: Phân lớp Naive Bayes
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Phân loại mức độ hài lòng khách hàng (satisfied/neutral/unsatisfied)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/final/clustered_data.csv')
customer_df = df[df['speaker'] == 'customer'].copy()
print(f'Customer messages: {len(customer_df)}')
print(f'Label distribution:')
print(customer_df['final_satisfaction'].value_counts())

## 4.1 Chuẩn bị dữ liệu

In [ ]:
# Gộp tất cả message customer theo conv_id
conv_text = customer_df.groupby('conv_id').agg(
    text=('message_clean', lambda x: ' '.join(x.dropna().astype(str))),
    satisfaction=('final_satisfaction', 'first')
).reset_index()

# Loại dòng trống
conv_text = conv_text[conv_text['text'].str.strip() != ''].reset_index(drop=True)
print(f'Tổng đoạn hội thoại: {len(conv_text)}')
print(f'Label distribution:')
print(conv_text['satisfaction'].value_counts())

X_text = conv_text['text']
y = conv_text['satisfaction']

## 4.2 TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=3000, min_df=2, max_df=0.95, ngram_range=(1, 2))
X = tfidf.fit_transform(X_text)
print(f'Feature matrix: {X.shape}')

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 4.3 Huấn luyện Multinomial Naive Bayes

In [ ]:
# Train
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test)

# Evaluate
print('=== NAIVE BAYES ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')
print()
print(classification_report(y_test, y_pred_nb))

## 4.4 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_nb, labels=['satisfied', 'neutral', 'unsatisfied'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['satisfied', 'neutral', 'unsatisfied'],
            yticklabels=['satisfied', 'neutral', 'unsatisfied'])
ax.set_xlabel('Dự đoán')
ax.set_ylabel('Thực tế')
ax.set_title('Confusion Matrix — Naive Bayes')
plt.tight_layout()
plt.savefig('../results/figures/03_classification/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 So sánh với mô hình khác

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(alpha=1.0),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42),
}

comparison = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    
    report = classification_report(y_test, y_pred, output_dict=True)
    comparison.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'CV Mean': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
        'F1 Macro': round(report['macro avg']['f1-score'], 4),
    })

comp_df = pd.DataFrame(comparison)
print('SO SÁNH MÔ HÌNH:')
print(comp_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(comp_df))
bars = ax.bar(x_pos, comp_df['Accuracy'], color=['#3498db', '#2ecc71', '#e74c3c'],
              edgecolor='black', width=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(comp_df['Model'])
ax.set_ylabel('Accuracy')
ax.set_title('So sánh Accuracy giữa các mô hình')
for bar, val in zip(bars, comp_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/03_classification/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Top từ quan trọng nhất mỗi lớp

In [ ]:
feature_names = tfidf.get_feature_names_out()
classes = nb_model.classes_

fig, axes = plt.subplots(1, len(classes), figsize=(18, 6))
for idx, cls in enumerate(classes):
    log_prob = nb_model.feature_log_prob_[idx]
    top_indices = log_prob.argsort()[-15:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    top_probs = [log_prob[i] for i in top_indices]
    
    axes[idx].barh(range(len(top_words)), top_probs, color=f'C{idx}', edgecolor='black')
    axes[idx].set_yticks(range(len(top_words)))
    axes[idx].set_yticklabels(top_words)
    axes[idx].invert_yaxis()
    axes[idx].set_title(f'{cls}')
    axes[idx].set_xlabel('Log Probability')

plt.suptitle('Top 15 từ quan trọng nhất mỗi lớp (Naive Bayes)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/03_classification/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Lưu kết quả

In [ ]:
# Lưu bảng so sánh
comp_df.to_csv('../results/tables/model_comparison.csv', index=False)
print('✅ Saved: model_comparison.csv')

# Lưu classification report
report_df = pd.DataFrame(classification_report(y_test, y_pred_nb, output_dict=True)).T
report_df.to_csv('../results/tables/classification_report.csv')
print('✅ Saved: classification_report.csv')

# Predict toàn bộ và lưu
all_pred = nb_model.predict(X)
conv_text['predicted_satisfaction'] = all_pred

# Merge về full df
full_df = pd.read_csv('../data/final/clustered_data.csv')
pred_map = dict(zip(conv_text['conv_id'], conv_text['predicted_satisfaction']))
full_df['predicted_satisfaction'] = full_df['conv_id'].map(pred_map)
full_df.to_csv('../data/final/classified_data.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: classified_data.csv')